In [1]:
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import List
from typing import Tuple
import numpy as np
import pandas as pd
from rdkit import DataStructs, Chem
from rdkit.Chem import AllChem, MolFromSmiles
from typing import Dict
from rdkit.Chem.Scaffolds.MurckoScaffold import MurckoScaffoldSmiles

from notebooks.utils.common import get_fragment_cost, get_path_cost_proxy
from rgfn.gfns.reaction_gfn.proxies.path_cost_proxy import PathCostProxy

/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: dlopen(/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/libpyg.so, 0x0006): Library not loaded: /Library/Frameworks/Python.framework/Versions/3.11/Python
  Referenced from: <75FFC412-93B5-322B-8E6D-268DA3498CF4> /Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/libpyg.so
  Reason: tried: '/Library/Frameworks/Python.framework/Versions/3.11/Python' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/Python.framework/Versions/3.11/Python' (no such file), '/Library/Frameworks/Python.framework/Versions/3.11/Python' (no such file)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'to

In [2]:
task_name = 'seh'
subset = 2
lower_threshold = True

In [3]:

use_cache = True
model_names = ['rgfn_old']
if not lower_threshold:
    threshold = 8.0 if task_name.lower() == 'seh' else 0.8
else:
    threshold = 7.2 if task_name.lower() == 'seh' else 0.7
if subset == 0:
    templates_list = ['rgfn_new_filtered']
elif subset == 1:
    templates_list = ['synflow_64']
elif subset == 2:
    templates_list = ['synflow_2_128']
else:
    templates_list = ['rgfn_new_filtered', 'synflow_64', 'synflow_2_128']
modes_similarity_threshold = 0.5
modes_every_n_iteration = 1000
n_cheapest_scaffolds = 100
n_forward_in_batch = 64

In [4]:
import pickle


In [5]:
from notebooks.utils.common import _get_scaffold, _get_fp
from rgfn.gfns.reaction_gfn.api.data_structures import Molecule
from notebooks.utils.training_results import TrainingResults
from more_itertools import chunked
from typing import Literal
from rdkit.Chem import MolToSmiles
from rdkit.Chem.Scaffolds.MurckoScaffold import GetScaffoldForMol, MakeScaffoldGeneric
from random import shuffle
from tqdm.notebook import tqdm


def get_scaffolds(result: TrainingResults, generic: bool = False):
    scaffolds = []
    for molecule in tqdm(result.molecules, desc='scaffolds'):
        scaffolds.append(_get_scaffold(molecule, generic=generic))
    return scaffolds


def get_num_scaffolds(result: TrainingResults, generic: bool = False):
    scaffolds = set()
    num_scaffolds = []
    scaffolds_list = result.scaffolds_generic if generic else result.scaffolds
    for idx, (scaffold, reward) in tqdm(enumerate(zip(scaffolds_list, result.rewards)), total=len(result.molecules),
                                        desc='num_scaffolds'):
        if reward > threshold:
            scaffolds.add(scaffold)
        num_scaffolds.append(len(scaffolds))
    return np.array(num_scaffolds)


def get_num_modes(result: TrainingResults,
                  molecule_transform: Literal[None, "scaffold", "scaffold_generic"],
                  similarity_threshold: float = modes_similarity_threshold,
                  fp_type: str = 'morgan_3'):
    interesting_molecules: Dict[str, float] = {}
    num_modes = []
    molecules = result.molecules
    if molecule_transform == 'scaffold':
        molecules = result.scaffolds
    elif molecule_transform == 'scaffold_generic':
        molecules = result.scaffolds_generic

    for idx, (molecule, reward) in (pbar := tqdm(enumerate(zip(molecules, result.rewards), start=1),
                                                 total=len(result.molecules),
                                                 desc='num_modes')):
        if reward > threshold:
            interesting_molecules[molecule] = max(interesting_molecules.get(molecule, 0), reward)
        if idx % (n_forward_in_batch * modes_every_n_iteration) == 0:
            sorted_molecules = [m[0] for m in sorted(interesting_molecules.items(), key=lambda x: x[1], reverse=True)]
            modes_fp_list = []

            def _is_new_mode(fp) -> bool:
                for fp_list in chunked(modes_fp_list, 1000):
                    similarities = DataStructs.BulkTanimotoSimilarity(fp, fp_list)
                    if max(similarities) > similarity_threshold:
                        return False
                return True

            for smiles in sorted_molecules:
                fp = _get_fp(smiles, fp_type=fp_type)
                if _is_new_mode(fp):
                    modes_fp_list.append(fp)
                    pbar.set_postfix({'num_modes': len(modes_fp_list)})
            num_modes.append(len(modes_fp_list))

    return np.array(num_modes)


def get_moving_average_path_costs(result: TrainingResults):
    averages = []
    current_average = 0
    for idx, (cost, reward) in tqdm(enumerate(zip(result.path_costs, result.rewards)), total=len(result.path_costs),
                                    desc='moving_average_path_costs'):
        if reward > threshold:
            current_average = (current_average * idx + cost) / (idx + 1)
        averages.append(current_average)
    return np.array(averages)


def get_last_n_average_path_costs(result: TrainingResults, n: int):
    averages = []
    for idx in tqdm(range(len(result.path_costs)), desc='last_n_average_path_costs'):
        costs = result.path_costs[max(0, idx - n):idx]
        current_average = np.mean(costs)
        averages.append(current_average)
    return np.array(averages)


def get_cheapest_scaffolds_path_costs(result: TrainingResults, generic: bool = False):
    scaffold_to_cost = defaultdict(lambda: np.inf)
    current_value = 0
    cost_list = []
    scaffolds = result.scaffolds_generic if generic else result.scaffolds
    for scaffold, cost, reward in tqdm(zip(scaffolds, result.path_costs, result.rewards),
                                       total=len(result.molecules), desc='cheapest_scaffolds_path_costs'):
        if reward > threshold:
            scaffold_to_cost[scaffold] = min(scaffold_to_cost[scaffold], cost)
            current_value = np.mean(sorted(scaffold_to_cost.values())[:n_cheapest_scaffolds])
        cost_list.append(current_value)
    return np.array(cost_list)


In [6]:
from rgfn import ReactionDataFactory




In [7]:
from notebooks.utils.common import get_path_costs
from notebooks.utils.training_results import TrainingResultsList

results_dict = {}

for templates_name in templates_list:
    path_cost_proxy = None
    sanitized_path_cost_proxy = None
    rxnflow_path_cost_proxy = None
    results_list = []
    for model_name in tqdm(model_names):
        sub_list = []
        for seed in range(10):
            try:
                result = TrainingResults(model_name=model_name,
                                         seed=seed,
                                         templates_name=templates_name,
                                         task_name=task_name,
                                         threshold=threshold,
                                         use_cache=use_cache)
            except FileNotFoundError as e:
                print(f"File not found: {e}")
                continue
            print(f"Processing {result.model_name} {result.seed}")
            if result.path_costs is None:
                if 'synflow' in result.model_name:
                    sanitized_path_cost_proxy = sanitized_path_cost_proxy or get_path_cost_proxy(templates_name,
                                                                                                 sanitized=True)
                    actual_path_cost_proxy = sanitized_path_cost_proxy
                elif 'rxnflow' in result.model_name:
                    rxnflow_path_cost_proxy = rxnflow_path_cost_proxy or get_path_cost_proxy(templates_name,
                                                                                             sanitized="rxnflow")
                    actual_path_cost_proxy = rxnflow_path_cost_proxy
                else:
                    path_cost_proxy = path_cost_proxy or get_path_cost_proxy(templates_name, sanitized=False)
                    actual_path_cost_proxy = path_cost_proxy
                result.load_heavy_stuff()
                result.path_costs, result.molecule_to_cheapest_cost = get_path_costs(result, actual_path_cost_proxy)
                if 'dyn' not in model_name:
                    result.molecule_to_cheapest_cost = None
            if result.last_n_average_path_costs is None:
                result.last_n_average_path_costs = get_last_n_average_path_costs(result, 500)
            if result.scaffolds is None:
                result.load_heavy_stuff()
                result.scaffolds = get_scaffolds(result, generic=False)
            if result.num_scaffolds is None:
                result.load_heavy_stuff()
                result.num_scaffolds = get_num_scaffolds(result, generic=False)
            if result.num_modes is None:
                result.load_heavy_stuff()
                result.num_modes = get_num_modes(result,
                                                 molecule_transform=None)

            result.save()
            # sub_list.append(result)
    #     if sub_list:
    #         results_list.append(TrainingResultsList(sub_list))
    # 
    # results_dict[templates_name] = results_list

  0%|          | 0/1 [00:00<?, ?it/s]

Processing rgfn_old 0
Using 128000 fragments, 105 reactions, and 197 anchored reactions
Cost mean and variance (193.19768125311134, 149.11302272980473)
Cost max 999.9285634452599



path_costs: 100%|██████████| 256064/256064 [00:01<00:00, 201622.02it/s]


last_n_average_path_costs:   0%|          | 0/256064 [00:00<?, ?it/s]

/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


...

/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


scaffolds:   0%|          | 0/256064 [00:00<?, ?it/s]

num_scaffolds:   0%|          | 0/256064 [00:00<?, ?it/s]

num_modes:   0%|          | 0/256064 [00:00<?, ?it/s]

File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_3.csv
File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_4.csv
File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_5.csv
File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_6.csv
File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_7.csv
File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_8.csv
File not found: File not found: results/synflow_2_128/seh/rgfn_old/training/paths_9.csv
